In [ ]:
# Run this cell first. If you see "Kernel ready" below, the kernel is working.
print("Kernel ready")

In [ ]:
# Project root / sys.path setup (same pattern as 01_data_exploration)
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root on sys.path:", PROJECT_ROOT)

### 1. Load WR weekly data and build modeling frame

In this notebook we:
- use the reusable helpers under `src/wr_predictor` to build a WR-week dataset
- define a **next-week PPR** target
- engineer rolling usage features
- train and evaluate a few baseline models.

The core dataset construction lives in Python modules (not here), so this notebook focuses on experimentation and model comparison.

In [ ]:
import polars as pl

from src.wr_predictor.dataset_builder import build_training_dataset

# Seasons configuration
# Example: keep one or more recent seasons as hold-out later if desired.
TRAINING_SEASONS = list(range(2015, 2025))  # 2015-2024 inclusive

wr_weekly = build_training_dataset(
    seasons=TRAINING_SEASONS,
    min_games_for_player=0,  # keep all rows; we will filter later as needed
)

print(wr_weekly.shape)
wr_weekly.head()

In [ ]:
# Restrict columns for modeling: identifiers, context, and numeric features

id_cols = [
    col for col in ["player_id", "player_name", "season", "week", "team", "opponent_team", "home_away"]
    if col in wr_weekly.columns
]

target_col = "next_week_ppr_points"

feature_cols = [
    c
    for c in wr_weekly.columns
    if c not in id_cols and c != target_col
]

model_frame = wr_weekly.select(id_cols + feature_cols + [target_col])
model_frame.head()

### 2. Define next-week target and leakage note

The `dataset_builder.build_training_dataset` function already:
- computes PPR fantasy points for each WR-week (`ppr_points`)
- creates a **next-week PPR** target (`next_week_ppr_points`) by shifting over games per player
- drops rows where the next-week target is missing.

All features in this notebook are constructed from historical information **available before** the predicted game (no post-game stats or next-week outcomes are used as inputs).

In [ ]:
# Basic cleaning: drop rows with too little history for rolling windows

numeric_cols = [c for c in feature_cols + [target_col] if pl.datatypes.is_numeric(wr_weekly.schema.get(c))]

# Example rule: require non-null rolling-3 features when present
min_history_mask = None
for col in [c for c in model_frame.columns if c.endswith("_rolling_3")]:
    cond = model_frame[col].is_not_null()
    min_history_mask = cond if min_history_mask is None else (min_history_mask & cond)

if min_history_mask is not None:
    model_frame_clean = model_frame.filter(min_history_mask)
else:
    model_frame_clean = model_frame

# Drop any remaining nulls in numeric columns only
model_frame_clean = model_frame_clean.drop_nulls(subset=[c for c in numeric_cols if c in model_frame_clean.columns])

model_frame_clean.shape

### 3. Time-aware train / validation / test split

We split along the time axis (by season) instead of random rows to better mimic a real forecasting setup.

Example split here:
- **Train**: seasons 2015–2021
- **Validation**: season 2022
- **Test**: seasons 2023–2024

You can adjust these boundaries later if you want more recent data in training or a larger hold-out set.

In [ ]:
import numpy as np
import pandas as pd

# Convert Polars frame to Pandas for scikit-learn / Keras
pdf = model_frame_clean.to_pandas()

season_col = "season"

train_mask = pdf[season_col].between(2015, 2021)
val_mask = pdf[season_col] == 2022
test_mask = pdf[season_col].between(2023, 2024)

id_cols_pd = [c for c in id_cols if c in pdf.columns]

X_cols = [c for c in feature_cols if c in pdf.columns]
y_col = target_col

X_train, y_train = pdf.loc[train_mask, X_cols], pdf.loc[train_mask, y_col]
X_val, y_val = pdf.loc[val_mask, X_cols], pdf.loc[val_mask, y_col]
X_test, y_test = pdf.loc[test_mask, X_cols], pdf.loc[test_mask, y_col]

X_train.shape, X_val.shape, X_test.shape

### 4. Baseline linear model (Ridge)

We standardize numeric features and fit a regularized linear model (Ridge) to predict `next_week_ppr_points`.

Metrics: MAE, RMSE, and R² on train/validation/test splits, plus a simple predicted-vs-actual scatter plot on the test set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Simple grid of alpha values for Ridge
alphas = [0.1, 1.0, 10.0]

best_alpha = None
best_val_rmse = None
best_model = None

for alpha in alphas:
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha, random_state=0)),
        ]
    )
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    val_rmse = mean_squared_error(y_val, val_pred, squared=False)

    if best_val_rmse is None or val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = alpha
        best_model = model

print(f"Best alpha: {best_alpha}, validation RMSE: {best_val_rmse:.3f}")

# Evaluate on all splits using the best model
train_pred = best_model.predict(X_train)
val_pred = best_model.predict(X_val)
test_pred = best_model.predict(X_test)

for split_name, y_true, y_hat in [
    ("Train", y_train, train_pred),
    ("Validation", y_val, val_pred),
    ("Test", y_test, test_pred),
]:
    mae = mean_absolute_error(y_true, y_hat)
    rmse = mean_squared_error(y_true, y_hat, squared=False)
    r2 = r2_score(y_true, y_hat)
    print(f"{split_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R^2={r2:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual next-week PPR")
plt.ylabel("Predicted next-week PPR (Ridge)")
plt.title("Ridge baseline: predicted vs actual (test set)")
plt.tight_layout()
plt.show()

### 5. Simple neural network model (Keras)

We now train a small multi-layer perceptron on the same standardized numeric feature matrix.

The NN uses early stopping based on validation loss to avoid overfitting, and we compare its performance to the Ridge baseline on the test set.

In [ ]:
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# Standardize features for NN
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train)
X_val_scaled = scaler_nn.transform(X_val)
X_test_scaled = scaler_nn.transform(X_test)

input_dim = X_train_scaled.shape[1]

nn_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),
])

nn_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = nn_model.fit(
    X_train_scaled,
    y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1,
)

nn_test_pred = nn_model.predict(X_test_scaled).ravel()

nn_mae = mean_absolute_error(y_test, nn_test_pred)
nn_rmse = mean_squared_error(y_test, nn_test_pred, squared=False)
nn_r2 = r2_score(y_test, nn_test_pred)

print(f"NN Test: MAE={nn_mae:.3f}, RMSE={nn_rmse:.3f}, R^2={nn_r2:.3f}")

### 6. Simple feature importance / interpretability

For the linear model, we can inspect coefficients after standardization to see which features are most associated with higher next-week PPR.

(NN interpretability is more complex; for now we focus on coefficient magnitudes from the Ridge model.)

In [ ]:
# Feature importances for the Ridge model (standardized features)

ridge = best_model.named_steps["ridge"]
scaler = best_model.named_steps["scaler"]

coefs = pd.Series(ridge.coef_, index=X_cols)
coefs_abs = coefs.abs().sort_values(ascending=False)

coefs_abs.head(20)

### 7. Example: projections for a given future week

This section sketches how you would generate projections for a new target week using the trained model.

To keep the notebook focused, we assume you will:
- construct a WR-week feature frame using only games **prior** to the target week
- apply the same feature engineering pipeline from `dataset_builder`
- standardize features with the same scaler used in training
- call `best_model.predict(...)` (or the NN) to obtain next-week PPR estimates.

Below is pseudocode-style scaffolding you can adapt once you have a specific target week and season in mind.

In [ ]:
# Sketch: building projections table for a specific (season, week)
# NOTE: This is a placeholder; adapt once you define your deployment workflow.

TARGET_SEASON = 2025  # example placeholder
TARGET_WEEK = 1

# You would:
# 1. Build a WR-week dataset covering games strictly before (TARGET_SEASON, TARGET_WEEK).
# 2. Apply the same feature engineering as in build_training_dataset.
# 3. Filter to rows corresponding to the last available game per WR before the target week.
# 4. Select the same feature columns X_cols, transform, and call best_model.predict.

# projections = some_frame.copy()
# projections_X = projections[X_cols]
# projections["pred_next_week_ppr_ridge"] = best_model.predict(projections_X)
# projections.sort_values("pred_next_week_ppr_ridge", ascending=False).head(50)

### 8. Notebook summary and next steps

**Setup and target**
- Target: `next_week_ppr_points` (next-week PPR fantasy points per WR-week).
- Data: WR weekly stats from `nflreadpy`, with lag and rolling features engineered via `src/wr_predictor.features` and targets via `dataset_builder`.
- Splits: train on seasons 2015–2021, validate on 2022, test on 2023–2024.

**Models and performance (to be filled in after runs)**
- Ridge baseline: note MAE / RMSE / R² from the printed metrics.
- Simple NN: compare test metrics and inspect where it helps or hurts vs Ridge.

**Where models struggle (qualitative observations)**
- High-volatility WRs with boom/bust usage.
- Small-sample players (few games or role changes).
- Injury- and matchup-driven swings that aren’t captured by simple features.

**Next experiments**
- More nuanced rolling windows (e.g., exponential decay, last-3 vs season-to-date splits).
- Encodings of opponent strength (defensive efficiency vs WRs, coverage types).
- Integrating `ffopportunity` expected-points metrics as extra inputs.
- Exploring gradient boosting models (e.g., XGBoost, LightGBM, HistGradientBoostingRegressor).